### Cell 1 - Check the GPU

In [ ]:
# Runtime > Change runtime type > T4 GPU  (do this BEFORE running)
!nvidia-smi --query-gpu=name,memory.total --format=csv

### Cell 2 - Install Ollama + ngrok client

In [ ]:
%%capture
!apt-get -qq install -y zstd pciutils lshw
!curl -fsSL https://ollama.com/install.sh | sh
!pip -q install pyngrok
print("installed")

### Cell 3 - Your settings (edit this one)

In [ ]:
# ---- EDIT THESE ----------------------------------------------------------
MODEL = "gpt-oss:20b"      # alternatives for a 15 GB T4: "qwen3:14b", "gemma3:12b", "qwen3:8b"
KEEP_ALIVE = "2h"          # how long the model stays loaded in GPU memory when idle
# ngrok token: free at https://dashboard.ngrok.com/get-started/your-authtoken
# Best: Colab left sidebar > key icon (Secrets) > add secret named NGROK_TOKEN > enable notebook access.
# ---------------------------------------------------------------------------
import os, getpass
try:
    from google.colab import userdata
    NGROK_TOKEN = userdata.get("NGROK_TOKEN")
except Exception:
    NGROK_TOKEN = None
if not NGROK_TOKEN:
    NGROK_TOKEN = getpass.getpass("Paste your ngrok authtoken: ").strip()
print("model:", MODEL, "| token loaded:", bool(NGROK_TOKEN))

### Cell 4 - Start the Ollama server

In [ ]:
import os, subprocess, time, urllib.request

env = dict(os.environ,
           OLLAMA_HOST="0.0.0.0:11434",
           OLLAMA_ORIGINS="*",
           OLLAMA_KEEP_ALIVE=KEEP_ALIVE,
           OLLAMA_FLASH_ATTENTION="1",
           OLLAMA_NUM_PARALLEL="1")
server = subprocess.Popen(["ollama", "serve"], env=env,
                          stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)

for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        print("Ollama is up")
        break
    except Exception:
        time.sleep(1)
else:
    print(open("/content/ollama.log").read()[-2000:])
    raise RuntimeError("Ollama did not start - see log above")

### Cell 5 - Download the model (first time takes a few minutes)

In [ ]:
!ollama pull {MODEL}
!ollama list

### Cell 6 - Warm up the model on the GPU

In [ ]:
import json, urllib.request, time
t = time.time()
req = urllib.request.Request(
    "http://127.0.0.1:11434/api/generate",
    data=json.dumps({"model": MODEL, "prompt": "Say OK.", "stream": False,
                     "options": {"num_predict": 8}, "keep_alive": KEEP_ALIVE}).encode(),
    headers={"Content-Type": "application/json"})
print(json.loads(urllib.request.urlopen(req, timeout=600).read())["response"].strip() or "(loaded)")
print(f"model ready in {time.time()-t:.0f}s")
!ollama ps

### Cell 7 - Open the ngrok tunnel and get your link

In [ ]:
import subprocess, time, json, urllib.request
from pyngrok import ngrok, conf

# clean up anything left from an earlier run of this cell
try: ngrok.kill()
except Exception: pass
subprocess.run(["pkill", "-f", "ngrok http"], check=False)
time.sleep(1)

ngrok.install_ngrok()
NGROK_BIN = conf.get_default().ngrok_path
subprocess.run([NGROK_BIN, "config", "add-authtoken", NGROK_TOKEN], check=True, capture_output=True)

# --host-header makes Ollama accept requests that arrive through the tunnel
tunnel_proc = subprocess.Popen(
    [NGROK_BIN, "http", "11434", "--host-header=localhost:11434", "--log=stdout"],
    stdout=open("/content/ngrok.log", "w"), stderr=subprocess.STDOUT)

PUBLIC_URL = None
for _ in range(30):
    try:
        data = json.loads(urllib.request.urlopen("http://127.0.0.1:4040/api/tunnels", timeout=2).read())
        for t in data.get("tunnels", []):
            if t["public_url"].startswith("https://"):
                PUBLIC_URL = t["public_url"]
        if PUBLIC_URL:
            break
    except Exception:
        pass
    time.sleep(1)
if not PUBLIC_URL:
    print(open("/content/ngrok.log").read()[-1500:])
    raise RuntimeError("ngrok did not start - see log above")

print("=" * 64)
print("  Your Nano AI server link:\n")
print("  ", PUBLIC_URL)
print("\n  In the nano-ai terminal run:  /url", PUBLIC_URL)
print("=" * 64)

### Cell 8 - Keep it alive (leave this cell running while you chat)

In [ ]:
# Colab disconnects idle sessions. This cell pings the server so the tab keeps working.
# Stop it with the square button when you are done (or just close the tab).
import time, urllib.request
print("Server running. Link:", PUBLIC_URL)
try:
    while True:
        try:
            urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=5)
            status = "ok"
        except Exception as e:
            status = f"PROBLEM: {e}"
        print(time.strftime("%H:%M:%S"), status, end="\r")
        time.sleep(60)
except KeyboardInterrupt:
    print("\nstopped")